# NP3-233-CD — Hourly Resource Outage Capacity: Endpoint Check

Pre-scraper review notebook. Confirms endpoint, schema, live/archive coverage,
and validates the dedup strategy before building the extractor.

**Dataset:** Reports hourly aggregated outage capacity by load zone, published every hour
with a 168-hour (7-day) forward window. Three mutually exclusive categories:
- **Total Resource outages** (cols C–F): routine maintenance + unexpected breakdowns → **KEEP**
- IRR outages (wind/solar) → drop
- New Equipment outages → drop

**Dedup rule (corrected — see Phase 6):** Each archive doc's 168-row window starts at the
*next hour* after posting. The hour-23 doc for date D contains only 1 row for D (hour 24).
To get all 24 hours of target date D, use the hour-23 doc from **D−1**. Archive CSV is
already system-wide (5 cols, no zone aggregation needed).

**Target range:** 2021-01-01 → present (must include Winter Storm Uri, Feb 2021).

Setup: `cp .env.example .env`, fill ERCOT credentials, then Run All.

In [1]:
import os, time, json
import requests
import pandas as pd
from datetime import date, timedelta
from dotenv import load_dotenv

load_dotenv()

_TOKEN_URL = (
    'https://ercotb2c.b2clogin.com'
    '/ercotb2c.onmicrosoft.com'
    '/B2C_1_PUBAPI-ROPC-FLOW'
    '/oauth2/v2.0/token'
)
CLIENT_ID = 'fec253ea-0d06-4272-a5e6-b478baeecd70'

_token_cache = {'token': None, 'expires_at': 0.0}

def _get_token() -> str:
    if _token_cache['token'] and time.time() < _token_cache['expires_at']:
        return _token_cache['token']
    resp = requests.post(_TOKEN_URL, data={
        'username'     : os.environ['ERCOT_USERNAME'],
        'password'     : os.environ['ERCOT_PASSWORD'],
        'grant_type'   : 'password',
        'client_id'    : CLIENT_ID,
        'scope'        : f'openid {CLIENT_ID} offline_access',
        'response_type': 'id_token',
    }, timeout=(10, 60))
    resp.raise_for_status()
    body = resp.json()
    _token_cache['token']      = body['id_token']
    _token_cache['expires_at'] = time.time() + int(body.get('expires_in', 3600)) - 60
    return _token_cache['token']

def _hdrs() -> dict:
    return {
        'Authorization'            : f'Bearer {_get_token()}',
        'Ocp-Apim-Subscription-Key': os.environ['ERCOT_SUBSCRIPTION_KEY'],
        'Accept'                   : 'application/json',
    }

def _get(url, **kwargs):
    r = requests.get(url, headers=_hdrs(), timeout=(10, 120), **kwargs)
    r.raise_for_status()
    return r.json()

def _to_df(body: dict) -> pd.DataFrame:
    fields = body.get('fields', [])
    cols = [f['name'] if isinstance(f, dict) else f for f in fields]
    return pd.DataFrame([dict(zip(cols, row)) for row in body.get('data', [])])

API_BASE = 'https://api.ercot.com/api/public-reports'
PRODUCT  = 'np3-233-cd'

print('Auth OK —', _get_token()[:20], '...')

Auth OK — eyJhbGciOiJSUzI1NiIs ...


---
## Phase 1 — Endpoint & Schema Discovery

In [2]:
meta_body = _get(f'{API_BASE}/{PRODUCT}')
print(f"Name:        {meta_body.get('name')}")
print(f"Description: {meta_body.get('description')[:200]}...")
print(f"Frequency:   {meta_body.get('generationFrequency')}")
print(f"First run:   {meta_body.get('firstRun')}")
print(f"Archive:     {meta_body.get('archiveDuration')} days")

artifact = meta_body['artifacts'][0]
ENDPOINT = artifact['_links']['endpoint']['href']
print(f"\nEndpoint: {ENDPOINT}")

for k, v in meta_body.get('_links', {}).items():
    print(f"Link [{k}]: {v['href']}")

Name:        Hourly Resource Outage Capacity
Description: This report includes all approved and accepted Planned, Forced and Maintenance Resource outages EXCEPT Resource outages for retirement of old equipment, seasonal mothballed (during the outaged season)...
Frequency:   Chron - Hourly
First run:   2013-10-20
Archive:     2555 days

Endpoint: https://api.ercot.com/api/public-reports/np3-233-cd/hourly_res_outage_cap
Link [self]: https://api.ercot.com/api/public-reports/np3-233-cd
Link [parent]: https://api.ercot.com/api/public-reports
Link [archive]: https://api.ercot.com/api/public-reports/archive/np3-233-cd
Link [bundle]: https://api.ercot.com/api/public-reports/bundle/np3-233-cd


In [3]:
probe = _get(ENDPOINT, params={'page': 1, 'size': 10})
fields = probe.get('fields', [])
meta   = probe.get('_meta', {})

print(f"totalPages: {meta.get('totalPages')}  totalRecords: {meta.get('totalRecords')}\n")
print(f"{'#':<4} {'Column':<45} {'Dtype':<12} {'Label'}")
print('-' * 100)
for i, f in enumerate(fields):
    name  = f.get('name') if isinstance(f, dict) else f
    dtype = f.get('dataType', '?') if isinstance(f, dict) else '?'
    label = f.get('label', '') if isinstance(f, dict) else ''
    print(f"{i:<4} {name:<45} {dtype:<12} {label}")

df_sample = _to_df(probe)
print('\nSample data:')
display(df_sample)

totalPages: 465694  totalRecords: 4656936

#    Column                                        Dtype        Label
----------------------------------------------------------------------------------------------------
0    postedDatetime                                DATETIME     Posted
1    operatingDate                                 DATE         Operating Date
2    hourEnding                                    INTEGER      Hour Ending
3    totalResourceMWZoneSouth                      INTEGER      Total Resource MW Zone South
4    totalResourceMWZoneNorth                      INTEGER      Total Resource MW Zone North
5    totalResourceMWZoneWest                       INTEGER      Total Resource MW Zone West
6    totalResourceMWZoneHouston                    INTEGER      Total Resource MW Zone Houston
7    totalIRRMWZoneSouth                           INTEGER      Total IRR MW Zone South
8    totalIRRMWZoneNorth                           INTEGER      Total IRR MW Zone North
9    totalI

,postedDatetime,operatingDate,hourEnding,totalResourceMWZoneSouth,totalResourceMWZoneNorth,totalResourceMWZoneWest,totalResourceMWZoneHouston,totalIRRMWZoneSouth,totalIRRMWZoneNorth,totalIRRMWZoneWest,totalIRRMWZoneHouston,totalNewEquipResourceMWZoneSouth,totalNewEquipResourceMWZoneNorth,totalNewEquipResourceMWZoneWest,totalNewEquipResourceMWZoneHouston
0,2026-09-17T16:00:47,2026-09-24,1,1910,2916,468,1376,809,1008,2677,97,524,1875,190,611
1,2026-09-17T16:00:47,2026-09-24,2,1910,2916,468,1376,809,1008,2677,97,524,1875,190,611
2,2026-09-17T16:00:47,2026-09-24,3,1910,2916,468,1376,809,1008,2677,97,524,1875,190,611
3,2026-09-17T16:00:47,2026-09-24,4,1910,2916,468,1376,809,1008,2677,97,524,1875,190,611
4,2026-09-17T16:00:47,2026-09-24,5,1910,2916,468,1376,809,1008,2677,97,524,1875,190,611
5,2026-09-17T16:00:47,2026-09-24,6,1910,2916,468,1376,809,1008,2677,97,524,1875,190,611
6,2026-09-17T16:00:47,2026-09-24,7,1910,2916,468,1376,809,1008,2677,97,524,1875,190,611
7,2026-09-17T16:00:47,2026-09-24,8,1919,2916,468,1376,809,1008,2677,97,524,1875,190,611
8,2026-09-17T16:00:47,2026-09-24,9,1919,2892,390,1376,809,914,2448,97,524,1875,190,611
9,2026-09-17T16:00:47,2026-09-24,10,1929,2892,390,1376,809,914,2448,97,524,1875,190,611


---
## Phase 2 — Live-API History Depth

Probe `totalRecords` for one-day windows at several past dates.
First date with records > 0 ≈ earliest live coverage.

In [4]:
PROBE_DATES = ['2021-01-15', '2021-02-15', '2022-01-15', '2023-01-15',
               '2023-06-15', '2023-12-01', '2023-12-10', '2023-12-15',
               '2024-06-15', '2025-06-15', '2026-06-15']

depth_rows = []
for pd_date in PROBE_DATES:
    try:
        body = _get(ENDPOINT, params={
            'operatingDateFrom': pd_date,
            'operatingDateTo':   pd_date,
            'page': 1, 'size': 1,
        })
        n = body.get('_meta', {}).get('totalRecords', 0)
    except requests.HTTPError as e:
        n = f'ERR {e.response.status_code}'
    depth_rows.append({'date': pd_date, 'totalRecords': n})
    time.sleep(0.5)

depth_df = pd.DataFrame(depth_rows)
display(depth_df)
print('\nFirst row with records > 0 ≈ earliest live coverage.')
print('Uri dates (2021-02) must come from the archive API.')

,date,totalRecords
0,2021-01-15,0
1,2021-02-15,0
2,2022-01-15,0
3,2023-01-15,0
4,2023-06-15,0
5,2023-12-01,0
6,2023-12-10,9
7,2023-12-15,2544
8,2024-06-15,4608
9,2025-06-15,4608



First row with records > 0 ≈ earliest live coverage.
Uri dates (2021-02) must come from the archive API.


---
## Phase 3 — Archive-API Depth & Doc Count

Check archive coverage and count docs in our target range (2021-01-01 → 2023-11-30).

In [5]:
ARCHIVE_URL = f'{API_BASE}/archive/{PRODUCT}'

# Overall archive stats
first_page = _get(ARCHIVE_URL, params={'page': 1, 'size': 1})
total_docs  = first_page['_meta']['totalRecords']
total_pages = first_page['_meta']['totalPages']
newest = first_page['archives'][0]

time.sleep(0.5)
last_page = _get(ARCHIVE_URL, params={'page': total_pages, 'size': 1})
oldest = last_page['archives'][0]

print(f'Total archive docs: {total_docs:,}')
print(f'Newest: {newest["postDatetime"]}')
print(f'Oldest: {oldest["postDatetime"]}')

time.sleep(0.5)

# Count docs in target range (2021-01-01 → 2023-12-10)
range_page = _get(ARCHIVE_URL, params={
    'postDatetimeFrom': '2021-01-01',
    'postDatetimeTo':   '2023-12-10',
    'page': 1, 'size': 1,
})
range_docs = range_page['_meta']['totalRecords']
print(f'\nDocs in 2021-01-01 → 2023-12-10: {range_docs:,}')

# Runtime estimate for archive
batch_size = 100
avg_batch_sec = 10
est_min = (range_docs / batch_size) * avg_batch_sec / 60
print(f'Estimated archive extraction time: {est_min:.0f} min ({est_min/60:.1f} hrs) at {batch_size} docs/batch')

Total archive docs: 108,512
Newest: 2026-09-17T16:00:47.000
Oldest: 2014-05-01T00:00:40.455



Docs in 2021-01-01 → 2023-12-10: 25,746
Estimated archive extraction time: 43 min (0.7 hrs) at 100 docs/batch


In [6]:
# Verify archive coverage across the FULL analysis range (2021-01-01 → 2023-12-10)
# Spot-check doc availability at the range start, Uri, and one date per year
import io, zipfile

SPOT_CHECK_RANGES = [
    ('2021-01-01', '2021-01-07', 'Range start (Jan 2021)'),
    ('2021-02-13', '2021-02-20', 'Winter Storm Uri'),
    ('2022-01-01', '2022-01-07', 'Mid-range (Jan 2022)'),
    ('2023-01-01', '2023-01-07', 'Mid-range (Jan 2023)'),
    ('2023-11-25', '2023-12-01', 'Range end (Nov-Dec 2023)'),
]

for dt_from, dt_to, label in SPOT_CHECK_RANGES:
    body = _get(ARCHIVE_URL, params={
        'postDatetimeFrom': dt_from,
        'postDatetimeTo':   dt_to,
        'page': 1, 'size': 1,
    })
    n = body['_meta']['totalRecords']
    status = '✓' if n > 0 else '⚠ MISSING'
    print(f'{status}  {label:30s}  {dt_from} → {dt_to}:  {n:,} docs')
    time.sleep(0.5)

print('\nAll spot-checks should show ✓. Any ⚠ MISSING means a gap in archive coverage.')

✓  Range start (Jan 2021)          2021-01-01 → 2021-01-07:  144 docs


✓  Winter Storm Uri                2021-02-13 → 2021-02-20:  168 docs


✓  Mid-range (Jan 2022)            2022-01-01 → 2022-01-07:  144 docs


✓  Mid-range (Jan 2023)            2023-01-01 → 2023-01-07:  144 docs


✓  Range end (Nov-Dec 2023)        2023-11-25 → 2023-12-01:  144 docs



All spot-checks should show ✓. Any ⚠ MISSING means a gap in archive coverage.


In [7]:
# Download one archive doc from the START of our range (Jan 2021) to inspect CSV schema
start_docs = _get(ARCHIVE_URL, params={
    'postDatetimeFrom': '2021-01-01',
    'postDatetimeTo':   '2021-01-07',
    'page': 1, 'size': 1,
})
sample_doc = start_docs['archives'][0]
print(f'Sample doc: postDatetime={sample_doc["postDatetime"]}, docId={sample_doc["docId"]}')

time.sleep(0.5)
dl_resp = requests.post(
    f'{ARCHIVE_URL}/download',
    headers=_hdrs(),
    json={'docIds': [sample_doc['docId']]},
    timeout=(10, 120),
)
dl_resp.raise_for_status()

# Archive uses nested ZIPs: outer ZIP → inner ZIP → CSV
with zipfile.ZipFile(io.BytesIO(dl_resp.content)) as outer_zf:
    print(f'Outer ZIP contents: {outer_zf.namelist()}')
    inner_zips = [n for n in outer_zf.namelist() if n.endswith('.zip')]
    csv_files  = [n for n in outer_zf.namelist() if n.endswith('.csv')]
    
    if csv_files:
        df_arch = pd.read_csv(outer_zf.open(csv_files[0]))
    elif inner_zips:
        inner_buf = io.BytesIO(outer_zf.read(inner_zips[0]))
        with zipfile.ZipFile(inner_buf) as inner_zf:
            print(f'Inner ZIP contents: {inner_zf.namelist()}')
            csv_name = [n for n in inner_zf.namelist() if n.endswith('.csv')][0]
            df_arch = pd.read_csv(inner_zf.open(csv_name))
    else:
        raise ValueError(f'No CSV or inner ZIP found: {outer_zf.namelist()}')
    
    print(f'\nArchive CSV columns ({len(df_arch.columns)}):')
    for i, col in enumerate(df_arch.columns):
        sample_val = df_arch[col].dropna().iloc[0] if not df_arch[col].dropna().empty else '—'
        print(f'  {i:<4} {col:<45} {str(df_arch[col].dtype):<12} {sample_val}')
    print(f'\nShape: {df_arch.shape}')
    print(f'Date range in this doc: {df_arch["Date"].min()} → {df_arch["Date"].max()}')
    display(df_arch.head(10))

Sample doc: postDatetime=2021-01-06T23:01:10.455, docId=752261754


Outer ZIP contents: ['752261754.cdr.00013103.0000000000000000.20210106.230110455.HRLYRESOUTCAPNP3233_csv.zip']
Inner ZIP contents: ['cdr.00013103.0000000000000000.20210106.230110.HRLYRESOUTCAPNP3233.csv']

Archive CSV columns (5):
  0    Date                                          str          01/06/2021
  1    HourEnding                                    int64        24
  2    TotalResourceMW                               int64        7521
  3    TotalIRRMW                                    int64        2498
  4    TotalNewEquipResourceMW                       int64        3159

Shape: (168, 5)
Date range in this doc: 01/06/2021 → 01/13/2021


,Date,HourEnding,TotalResourceMW,TotalIRRMW,TotalNewEquipResourceMW
0,01/06/2021,24,7521,2498,3159
1,01/07/2021,1,7521,2498,3159
2,01/07/2021,2,7521,2498,3159
3,01/07/2021,3,7521,2498,3159
4,01/07/2021,4,7521,2498,3159
5,01/07/2021,5,7521,2498,3159
6,01/07/2021,6,7521,2498,3159
7,01/07/2021,7,7797,2498,3159
8,01/07/2021,8,7807,2777,3159
9,01/07/2021,9,7767,2622,3159


---
## Phase 4 — Posting Pattern & Hour-23 Dedup Validation

Confirm that the hour-23 report exists for each day and covers all 24 hourEnding values.

In [8]:
# Fetch a full day from live API to study posting pattern
TEST_DATE = '2026-09-15'
day_body = _get(ENDPOINT, params={
    'operatingDateFrom': TEST_DATE,
    'operatingDateTo':   TEST_DATE,
    'page': 1, 'size': 5000,
})
df_day = _to_df(day_body)
print(f'Records for {TEST_DATE}: {len(df_day)}')
print(f'Unique postedDatetime: {df_day["postedDatetime"].nunique()}')
print(f'Unique hourEnding: {sorted(df_day["hourEnding"].unique())}')

# Show posting frequency by hour
df_day['posted_dt'] = pd.to_datetime(df_day['postedDatetime'])
df_day['posted_date'] = df_day['posted_dt'].dt.strftime('%Y-%m-%d')
df_day['posted_hour'] = df_day['posted_dt'].dt.hour

print(f'\nReports by posted_date:')
for d, grp in df_day.groupby('posted_date'):
    print(f'  {d}: {len(grp)} rows from {grp["posted_hour"].min()}:00 to {grp["posted_hour"].max()}:00')

Records for 2026-09-15: 4608
Unique postedDatetime: 216
Unique hourEnding: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]

Reports by posted_date:
  2026-09-08: 552 rows from 0:00 to 23:00
  2026-09-09: 576 rows from 0:00 to 23:00
  2026-09-10: 576 rows from 0:00 to 23:00
  2026-09-11: 576 rows from 0:00 to 23:00
  2026-09-12: 576 rows from 0:00 to 23:00
  2026-09-13: 576 rows from 0:00 to 23:00
  2026-09-14: 576 rows from 0:00 to 23:00
  2026-09-15: 576 rows from 0:00 to 23:00
  2026-09-16: 24 rows from 0:00 to 23:00


In [9]:
# Validate hour-23 dedup: filter to the report posted at hour 23 of the operatingDate
mask_h23 = (df_day['posted_date'] == TEST_DATE) & (df_day['posted_hour'] == 23)
df_h23 = df_day[mask_h23].copy()

print(f'Hour-23 report rows: {len(df_h23)}')
print(f'hourEnding coverage: {sorted(df_h23["hourEnding"].unique())}')
assert len(df_h23) == 24, f'Expected 24 rows, got {len(df_h23)}'
assert set(df_h23['hourEnding']) == set(range(1, 25)), 'Missing hourEnding values'
print('✓ Hour-23 dedup produces exactly 24 rows (hourEnding 1-24)')

# Show the deduped output
KEEP = ['operatingDate', 'hourEnding', 'postedDatetime',
        'totalResourceMWZoneSouth', 'totalResourceMWZoneNorth',
        'totalResourceMWZoneWest', 'totalResourceMWZoneHouston']
df_h23_clean = df_h23[KEEP].sort_values('hourEnding').reset_index(drop=True)
df_h23_clean['totalResourceMWSystemWide'] = (
    df_h23_clean['totalResourceMWZoneSouth'] +
    df_h23_clean['totalResourceMWZoneNorth'] +
    df_h23_clean['totalResourceMWZoneWest'] +
    df_h23_clean['totalResourceMWZoneHouston']
)
display(df_h23_clean)
print(f'\nSystem-wide outage range: {df_h23_clean["totalResourceMWSystemWide"].min()} – {df_h23_clean["totalResourceMWSystemWide"].max()} MW')

Hour-23 report rows: 24
hourEnding coverage: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]
✓ Hour-23 dedup produces exactly 24 rows (hourEnding 1-24)


,operatingDate,hourEnding,postedDatetime,totalResourceMWZoneSouth,totalResourceMWZoneNorth,totalResourceMWZoneWest,totalResourceMWZoneHouston,totalResourceMWSystemWide
0,2026-09-15,1,2026-09-15T23:00:47,2696,4076,275,2218,9265
1,2026-09-15,2,2026-09-15T23:00:47,2696,4076,275,2218,9265
2,2026-09-15,3,2026-09-15T23:00:47,2696,4076,275,2218,9265
3,2026-09-15,4,2026-09-15T23:00:47,2796,4013,290,2218,9317
4,2026-09-15,5,2026-09-15T23:00:47,2796,4013,290,2218,9317
5,2026-09-15,6,2026-09-15T23:00:47,2796,4013,290,2160,9259
6,2026-09-15,7,2026-09-15T23:00:47,2796,4084,290,2160,9330
7,2026-09-15,8,2026-09-15T23:00:47,2889,4102,290,2160,9441
8,2026-09-15,9,2026-09-15T23:00:47,2949,4024,297,2160,9430
9,2026-09-15,10,2026-09-15T23:00:47,2949,4813,297,2160,10219



System-wide outage range: 9259 – 10553 MW


---
## Phase 5 — Volume & Runtime Estimate

In [10]:
# Live API volume (Dec 2023 → present)
live_body = _get(ENDPOINT, params={
    'operatingDateFrom': '2023-12-10',
    'page': 1, 'size': 1,
})
live_total = live_body['_meta']['totalRecords']
live_pages_1k = live_total // 1000 + 1
live_time_hr = live_pages_1k * 1.0 / 3600  # 1s per page

print('=== LIVE API (2023-12-10 → present) ===')
print(f'Total records: {live_total:,}')
print(f'Pages at size=1000: {live_pages_1k:,}')
print(f'Est. time: {live_time_hr:.1f} hrs → Method {"B" if live_time_hr < 0.17 else "C"}')

time.sleep(0.5)

# Archive volume (2021-01-01 → 2023-12-09)
arch_range = _get(ARCHIVE_URL, params={
    'postDatetimeFrom': '2021-01-01',
    'postDatetimeTo':   '2023-12-10',
    'page': 1, 'size': 1,
})
arch_docs = arch_range['_meta']['totalRecords']
arch_batches = arch_docs // 100 + 1
arch_time_min = arch_batches * 10 / 60  # ~10s per batch of 100

print(f'\n=== ARCHIVE (2021-01-01 → 2023-12-09) ===')
print(f'Total docs: {arch_docs:,}')
print(f'Batches at 100 docs/batch: {arch_batches:,}')
print(f'Est. time: {arch_time_min:.0f} min ({arch_time_min/60:.1f} hrs) → Method {"B" if arch_time_min < 10 else "C"}')

# After dedup: expected final row count
from datetime import date as dt_date
total_days = (dt_date.today() - dt_date(2021, 1, 1)).days
final_rows = total_days * 24
print(f'\n=== DEDUPED OUTPUT ===')
print(f'Days: {total_days} × 24 hrs = {final_rows:,} rows')
print(f'Est. parquet size: {final_rows * 50 / 1e6:.1f} MB (snappy)')

=== LIVE API (2023-12-10 → present) ===
Total records: 4,656,936
Pages at size=1000: 4,657
Est. time: 1.3 hrs → Method C



=== ARCHIVE (2021-01-01 → 2023-12-09) ===
Total docs: 25,746
Batches at 100 docs/batch: 258
Est. time: 43 min (0.7 hrs) → Method C

=== DEDUPED OUTPUT ===
Days: 2085 × 24 hrs = 50,040 rows
Est. parquet size: 2.5 MB (snappy)


---
## Phase 6 — Archive 168-Row Window & Corrected Dedup Rule

The hour-23 dedup validated in Phase 4 used the **live API**, which indexes by
`operatingDate`. Archive docs work differently: each doc is a 168-row forward snapshot
starting at the **next hour** after posting.

**Problem:** the hour-23 doc posted on date D contains only **1 row** for D (hour 24) —
the other 167 rows cover D+1 through D+7.

**Fix:** for target date D, use the hour-23 doc from **D−1**, which contains all 24 hours
of D as rows 2–25 of its 168-row window.

Below: empirical verification on a normal day, spring-forward, and fall-back.

In [ ]:
# Demonstrate the problem: hour-23 doc for a day only has 1 row for that day
# (its 168-row window starts at hour 24 = the next hour after 23:00 posting)

normal_doc = _get(ARCHIVE_URL, params={
    'postDatetimeFrom': '2021-01-15T22:30',
    'postDatetimeTo':   '2021-01-15T23:30',
    'page': 1, 'size': 1,
})
doc = normal_doc['archives'][0]
print(f'Doc posted: {doc["postDatetime"]}')

time.sleep(0.5)
dl = requests.post(f'{ARCHIVE_URL}/download', headers=_hdrs(),
                   json={'docIds': [doc['docId']]}, timeout=(10, 120))
dl.raise_for_status()

with zipfile.ZipFile(io.BytesIO(dl.content)) as outer_zf:
    inner_zips = [n for n in outer_zf.namelist() if n.endswith('.zip')]
    csv_files  = [n for n in outer_zf.namelist() if n.endswith('.csv')]
    if csv_files:
        df_window = pd.read_csv(outer_zf.open(csv_files[0]))
    else:
        inner_buf = io.BytesIO(outer_zf.read(inner_zips[0]))
        with zipfile.ZipFile(inner_buf) as inner_zf:
            csv_name = [n for n in inner_zf.namelist() if n.endswith('.csv')][0]
            df_window = pd.read_csv(inner_zf.open(csv_name))

print(f'\n168-row window date distribution:')
print(df_window['Date'].value_counts().sort_index().to_string())
print(f'\nRows for posting date (01/15/2021): {len(df_window[df_window["Date"] == "01/15/2021"])}')
print('→ Only 1 row (hour 24). The full day is in the D−1 doc.')

---
## Decision Checklist

- [ ] Live API starts Dec 2023 — confirm archive covers 2021-01-01 (incl. Uri Feb 2021)
- [ ] Archive CSV column names match live API (or document renames needed)
- [ ] Hour-23 dedup produces exactly 24 rows per day
- [ ] Volume/runtime acceptable for two-source strategy
- [ ] KEEP_COLS confirmed: only Total Resource MW zone columns

**Next:** Write `ercot_outage_cap.py` extractor with `--trial-only` / `--auto` modes.